# The same system in LangChain
The module builds with LangChain. `rag.py` and `agent.py` are written without a framework so every step is visible; `jarvis.lc` rebuilds both from LangChain components over the same Chroma collections, prompts, scope rules and figure check:

* retrieval: an `EnsembleRetriever` fusing `langchain_chroma.Chroma` (vector) and `BM25Retriever` (keywords), both inside the question's scope and the caller's role, then `CrossEncoderReranker` and the neighbouring chunks
* chain (LCEL): rewrite | retrieve | prompt | chat model | retry once if declined | JSON | figure check
* engines: the embedding model and reranker on ONNX Runtime through LangChain's `Embeddings` and `BaseCrossEncoder` interfaces
* agent: `langchain.agents.create_agent` with the agent's tools as `StructuredTool`s, the role bound inside each tool
* models: `ChatBedrockConverse`, `ChatOpenAI` or local Ollama, chosen by `JARVIS_PROVIDER`

In [1]:
import sys, json, pandas as pd
sys.path.insert(0, '../src')
pd.set_option('display.max_colwidth', 120)

In [2]:
from jarvis.lc import LCRAG, lc_agent
lc = LCRAG()
out = lc.answer("What was TD's provision for credit losses in Q2 2025?")
{k: out[k] for k in ('answer', 'confidence', 'figure_check', 'sources', 'ms')}

{'answer': "TD's provision for credit losses in Q2 2025 was $1,341 million. This comprised $946 million in Stage 3 (impaired) provisions and $395 million in Stage 1 and Stage 2 (performing) provisions. [S2]",
 'confidence': 'high',
 'figure_check': [{'figure': '$1,341 million', 'supported': True},
  {'figure': '$946 million', 'supported': True},
  {'figure': '$395 million', 'supported': True}],
 'sources': [{'id': 'S2',
   'bank': 'TD',
   'document': 'Quarterly Report',
   'period': 'Q2 fiscal 2025',
   'section': 'How We Performed Corporate Overview',
   'page': 15}],
 'ms': 10312}

## LangChain against the native build on the Step 3 questions
Same 15 questions, same prompt (chain of thought), manual verdicts. The first LangChain chain searched by vector only and missed table rows (10 of 15); with BM25 keywords fused in an EnsembleRetriever, the rewrite, the neighbouring chunks and the retry, it reaches 14 of 15.

In [3]:
res = pd.DataFrame(json.load(open('../reports/step3_results.json')))
rev = pd.DataFrame(json.load(open('../reports/step3_review.json'))['reviews'])
m = res.merge(rev, on=['id', 'config'])
m = m[m.config.isin(['native:cot+v5', 'langchain:cot', 'langchain:cot+v4'])]
m.pivot(index='id', columns='config', values='verdict')

config,langchain:cot,langchain:cot+v4,native:cot+v5
id,,,
Q01,correct,wrong,correct
Q02,correct,correct,correct
Q03,correct,correct,correct
Q04,correct,correct,correct
Q05,correct,correct,correct
Q06,correct,correct,correct
Q07,correct,correct,correct
Q08,correct,correct,correct
Q09,correct,correct,correct


In [4]:
m.groupby('config').agg(correct=('verdict', lambda v: (v == 'correct').sum()), citations_accurate=('citations_accurate', 'sum'))

,correct,citations_accurate
config,,
langchain:cot,10,10
langchain:cot+v4,14,14
native:cot+v5,15,15


## The LangChain agent
The same question through both agents; the audit trail lists every tool call.

In [5]:
q = 'Compare the CET1 ratios of RBC and TD at the end of fiscal 2025.'
a = lc_agent(q)
print(a['answer'][:1200])
pd.DataFrame(a['tool_calls'])

Based on the fiscal 2025 annual reports:

**RBC CET1 Ratio:** 13.5% [T2:S1]

**TD CET1 Ratio:** 14.7% [T1:S3]

TD's CET1 ratio of 14.7% is higher than RBC's 13.5% at the end of fiscal 2025. TD's CET1 ratio exceeds RBC's by 1.2 percentage points.


,tool,args,result_chars
0,extract_metrics,"{'bank': 'RBC', 'fiscal_year': 2025, 'metrics': ['CET1']}",10993
1,extract_metrics,"{'bank': 'TD', 'fiscal_year': 2025, 'metrics': ['CET1']}",9947


Why the native build stays the default: on the same questions it scores 15 of 15 against 14 (the LangChain chain once quoted PCL on loans as total PCL), and it exposes every step to the model review committee in one place (follow up questions, one cost log). The LangChain build shows the same design in the framework the module teaches and switches provider with one setting.